# 7CS353 Machine Learning Lab — Assignment 9
### Clustering: K-Means and Hierarchical

Name: Atharv V Kulkarni
PRN: 245109074
Class: TY CSE(B)

## B1. Explore, Preprocess and Choose k

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.decomposition import PCA
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
np.random.seed(42)

In [ ]:
df = pd.read_csv("customers.csv")
print(df.shape)
print(df.dtypes)
print(df.head())
print(df.describe())
print(df.isnull().sum())

In [ ]:
sns.pairplot(df)
plt.show()

**Solution:**

_(fill in after viewing the pairplot — note whether any pair of features visually suggests natural groupings)_

In [ ]:
scaler = StandardScaler()
x_scaled = scaler.fit_transform(df[["age", "annual_spend", "purchase_freq"]])

**Solution:**

_(explain in 2-3 sentences why scaling is essential before clustering with Euclidean distance based algorithms like K-Means)_

In [ ]:
inertias = []
silhouettes = []
k_range = range(2, 9)
for k in k_range:
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    labels = km.fit_predict(x_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(x_scaled, labels))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13,5))
axes[0].plot(list(k_range), inertias, marker="o", color="royalblue")
axes[0].axvline(x=4, linestyle="--", color="gray")
axes[0].set_xlabel("k")
axes[0].set_ylabel("wcss (inertia)")
axes[0].set_title("elbow method")

axes[1].plot(list(k_range), silhouettes, marker="o", color="crimson")
axes[1].axvline(x=4, linestyle="--", color="gray")
axes[1].set_xlabel("k")
axes[1].set_ylabel("silhouette score")
axes[1].set_title("silhouette diagnostic")
plt.show()

**Solution:**

_(state your chosen k in >=2 sentences, citing evidence from the elbow bend and the silhouette peak above)_

## B2. K-Means from scratch + sklearn verification

In [ ]:
def kmeans_scratch(x, k, max_iter=300, seed=42):
    rng = np.random.default_rng(seed)
    n = x.shape[0]
    idx = rng.choice(n, k, replace=False)
    centroids = x[idx].copy()
    labels = np.zeros(n, dtype=int)
    for iteration in range(max_iter):
        distances = np.linalg.norm(x[:, None, :] - centroids[None, :, :], axis=2)
        new_labels = np.argmin(distances, axis=1)
        if np.array_equal(new_labels, labels) and iteration > 0:
            break
        labels = new_labels
        for i in range(k):
            members = x[labels == i]
            if len(members) > 0:
                centroids[i] = members.mean(axis=0)
    wcss = 0.0
    for i in range(k):
        members = x[labels == i]
        wcss += np.sum((members - centroids[i])**2)
    return labels, centroids, wcss

In [ ]:
labels_scratch, centroids_scratch, wcss_scratch = kmeans_scratch(x_scaled, k=4)
print("scratch wcss:", wcss_scratch)
print("scratch centroids:")
print(centroids_scratch)

In [ ]:
km_sklearn = KMeans(n_clusters=4, n_init=10, random_state=42)
labels_sklearn = km_sklearn.fit_predict(x_scaled)
print("sklearn inertia:", km_sklearn.inertia_)
pct_diff = abs(wcss_scratch - km_sklearn.inertia_) / km_sklearn.inertia_ * 100
print("percentage difference:", pct_diff)

In [ ]:
pca = PCA(n_components=2)
x_pca = pca.fit_transform(x_scaled)
centroids_pca = pca.transform(km_sklearn.cluster_centers_)

In [ ]:
fig, ax = plt.subplots(figsize=(7,6))
scatter = ax.scatter(x_pca[:,0], x_pca[:,1], c=labels_sklearn, cmap="viridis", alpha=0.75)
ax.scatter(centroids_pca[:,0], centroids_pca[:,1], c="red", marker="*", s=250, label="centroids")
ax.set_xlabel("pca component 1")
ax.set_ylabel("pca component 2")
ax.set_title("k-means clusters (k=4) in pca space")
ax.legend()
plt.show()

## B3. Hierarchical clustering and dendrogram

In [ ]:
z = linkage(x_scaled, method="ward")

In [ ]:
fig, ax = plt.subplots(figsize=(10,6))
dendrogram(z, truncate_mode="lastp", p=20, ax=ax)
ax.set_xlabel("sample index (or cluster size)")
ax.set_ylabel("linkage distance")
ax.set_title("ward linkage dendrogram")
plt.show()

**Solution:**

_(identify the largest vertical gap in the dendrogram above, draw where a horizontal cut would go, and state how many clusters that cut produces)_

In [ ]:
labels_hier = fcluster(z, t=4, criterion="maxclust")
sil_hier = silhouette_score(x_scaled, labels_hier)
db_hier = davies_bouldin_score(x_scaled, labels_hier)
print("hierarchical silhouette:", sil_hier)
print("hierarchical davies-bouldin:", db_hier)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13,5))
axes[0].scatter(x_pca[:,0], x_pca[:,1], c=labels_sklearn, cmap="viridis", alpha=0.75)
axes[0].set_xlabel("pca component 1")
axes[0].set_ylabel("pca component 2")
axes[0].set_title("k-means clusters (k=4)")

axes[1].scatter(x_pca[:,0], x_pca[:,1], c=labels_hier, cmap="viridis", alpha=0.75)
axes[1].set_xlabel("pca component 1")
axes[1].set_ylabel("pca component 2")
axes[1].set_title("hierarchical ward clusters (k=4)")
plt.show()

**Solution:**

_(state whether the two clustering assignments look visually similar in pca space, and point out any region where they disagree)_

## B4. Cluster profiling and algorithm comparison

In [ ]:
df_profiled = df.copy()
df_profiled["cluster"] = labels_sklearn
profile = df_profiled.groupby("cluster").agg(
    size=("cluster", "count"),
    mean_age=("age", "mean"),
    mean_annual_spend=("annual_spend", "mean"),
    mean_purchase_freq=("purchase_freq", "mean")
)
print(profile)

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(8,12))
sns.boxplot(x="cluster", y="age", data=df_profiled, ax=axes[0])
axes[0].set_title("age by cluster")
sns.boxplot(x="cluster", y="annual_spend", data=df_profiled, ax=axes[1])
axes[1].set_title("annual spend by cluster")
sns.boxplot(x="cluster", y="purchase_freq", data=df_profiled, ax=axes[2])
axes[2].set_title("purchase frequency by cluster")
plt.tight_layout()
plt.show()

**Solution:**

_(give each cluster a descriptive label such as "occasional low-spenders" based on the profile table and box plots, with 1-2 sentences characterising each)_

In [ ]:
ch_kmeans = calinski_harabasz_score(x_scaled, labels_sklearn)
ch_hier = calinski_harabasz_score(x_scaled, labels_hier)
sil_kmeans = silhouette_score(x_scaled, labels_sklearn)
db_kmeans = davies_bouldin_score(x_scaled, labels_sklearn)
print("k-means -> silhouette:", sil_kmeans, "davies-bouldin:", db_kmeans, "calinski-harabasz:", ch_kmeans)
print("hierarchical -> silhouette:", sil_hier, "davies-bouldin:", db_hier, "calinski-harabasz:", ch_hier)

| Metric | K-Means | Hierarchical (Ward) |
|---|---|---|
| Silhouette | _ | _ |
| Davies-Bouldin | _ | _ |
| Calinski-Harabasz | _ | _ |

_(fill in from the printed values above, then justify in >=3 sentences which algorithm gives better-defined clusters)_

## B5. Take-home investigation

In [ ]:
km_raw = KMeans(n_clusters=4, n_init=10, random_state=42)
labels_raw = km_raw.fit_predict(df[["age", "annual_spend", "purchase_freq"]].values)
sil_raw = silhouette_score(df[["age", "annual_spend", "purchase_freq"]].values, labels_raw)
print("raw wcss:", km_raw.inertia_, "raw silhouette:", sil_raw)
print("scaled wcss:", km_sklearn.inertia_, "scaled silhouette:", sil_kmeans)

**Solution:**

_(explain the difference between the raw and scaled WCSS/silhouette results, referring to the different feature scales of age, annual_spend and purchase_freq)_

In [ ]:
dbscan = DBSCAN(eps=0.5, min_samples=5)
labels_dbscan = dbscan.fit_predict(x_scaled)
n_clusters_dbscan = len(set(labels_dbscan)) - (1 if -1 in labels_dbscan else 0)
n_noise = np.sum(labels_dbscan == -1)
print("dbscan clusters found:", n_clusters_dbscan)
print("dbscan noise points:", n_noise)

**Solution:**

_(state how many clusters and noise points DBSCAN found, and describe a scenario where DBSCAN would be preferred over K-Means)_

**Reflection:**

_(write >=4 sentences on the three most important decisions in a clustering workflow, and what you would do differently on a real business dataset)_